# All

In [ ]:
import os
import numpy as np
from contextlib import redirect_stdout
from tqdm import tqdm
from time import time
from collections import defaultdict
from tmu.models.autoencoder.autoencoder import TMAutoEncoder
from sklearn.metrics.pairwise import cosine_similarity
from evaluation import Evaluation
from tools import Tools
from directories import Dicrectories

target_similarity=defaultdict(list)
clause_weight_threshold = 0
number_of_examples = 100
clause_drop_p = 0.0
factor = 20
clauses = 80
T = factor*40
s = 5.0
accumulation = 10
epochs = 1000
max_spearman = 0.9

eval = Evaluation()
def preprocess_text(text):
    return text
vectorizer_X = Tools.read_pickle_data("vectorizer_X.pickle")
feature_names = vectorizer_X.get_feature_names_out()
number_of_features = vectorizer_X.get_feature_names_out().shape[0]

for dataset_name in os.listdir(Dicrectories.datasets):
    if dataset_name == 'rg-65':
        current_folder_path = os.path.join(Dicrectories.datasets, dataset_name)
        if os.path.isdir(current_folder_path):
            files_start_name = os.path.join(current_folder_path, dataset_name)

            pair_list = Tools.get_dataset_pairs(files_start_name + '.csv')
            output_active, target_words = Tools.get_targets(files_start_name)
            
            result_filepath = Dicrectories.test(dataset_name,"all_phase2")
            with open(result_filepath, 'w') as file, redirect_stdout(file):
                tm = TMAutoEncoder(clauses, T, s, output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=0.5)
                total_training = 0
                print("Epochs: %d" % epochs)
                print("Target words: %d" % len(target_words))
                print("Accumulation: %d" % accumulation)
                print("Examples: %d" % number_of_examples)
                print("No of features: %d" % number_of_features)
                print("Clauses: %d\n" % clauses)
                
                epochs_progress_bar = tqdm(total=epochs, desc="Running Epochs")
                for e in range(epochs):
                    print("\nEpoch #: %d" % e)
                    start_training = time()
                    tm.knowledge_fit(
                        number_of_examples = number_of_examples,
                        number_of_features = number_of_features,
                        top_max_clauses1 = 0,
                        top_max_clauses2 = 2,
                        with_clause_update = False,
                        print_c = False
                        )
                    stop_training = time()
                    epoch_time = stop_training - start_training
                    Tools.print_training_time(epoch_time)
                    total_training = total_training + epoch_time

                    profile = np.empty((len(target_words), clauses))
                    for i in range(len(target_words)):
                        weights = tm.get_weights(i)
                        profile[i,:] = np.where(weights >= clause_weight_threshold, weights, 0)
                    similarity = cosine_similarity(profile)
                    for i in range(len(target_words)):
                        sorted_index = np.argsort(-1*similarity[i,:])
                        for j in range(1, len(target_words)):
                            target_similarity[(target_words[i], target_words[sorted_index[j]])]  = similarity[i,sorted_index[j]]
                    spearman = eval.calculate(target_similarity,pair_list)
                    if spearman > max_spearman:
                        break
                    epochs_progress_bar.update(1)
                epochs_progress_bar.close()

                print("\n=====================================\nClauses\n=====================================")
                for j in range(clauses):
                    print("Clause #%-2d " % (j), end=' ')
                    for tw in range(len(target_words)):
                        print("%s:W%-5d " % (target_words[tw], tm.get_weight(tw, j)), end='| ')
                    l = [] 
                    number_of_literals = 0 
                    for k in range(tm.clause_bank.number_of_literals):
                        if tm.get_ta_action(j, k) == 1:
                            number_of_literals = number_of_literals + 1
                            if k < tm.clause_bank.number_of_features:
                                l.append("%s(%d)" % (feature_names[k], tm.clause_bank.get_ta_state(j, k)))
                            else:
                                l.append("¬%s(%d)" % (feature_names[k-tm.clause_bank.number_of_features], tm.clause_bank.get_ta_state(j, k)))
                    print(": No of features:%-6d" % (number_of_literals), end=" ==> ")
                    try:
                        print(" - ".join(l))
                    except UnicodeEncodeError:
                        print(" exception ")
                
                print("\n=====================================\nWord Similarity\n=====================================")
                max_word_length = len(max(target_words, key=len))
                list_of_words = []
                target_words_with_min_max = []
                for i in range(len(target_words)):
                    row_of_similarity = []
                    sorted_index = np.argsort(-1*similarity[i,:])
                    min_similarity = 1.0
                    max_similarity = 0.0
                    word_similarity = []
                    for j in range(1, len(target_words)):
                        target_similarity[(target_words[i], target_words[sorted_index[j]])]  = similarity[i,sorted_index[j]]
                        row_of_similarity.append(target_words[sorted_index[j]])
                        word_similarity.append("{:<{}}({:.2f})  ".format(target_words[sorted_index[j]], max_word_length, similarity[i, sorted_index[j]]))
                        if(min_similarity > similarity[i,sorted_index[j]]):
                            min_similarity = similarity[i,sorted_index[j]]
                        if(max_similarity < similarity[i,sorted_index[j]]):
                            max_similarity = similarity[i,sorted_index[j]]
                
                    output_line = f"{target_words[i]:<{max_word_length}}: Min:{min_similarity:.2f}, Max:{max_similarity:.2f}"
                    print(output_line, end='     ==> ')
                    print(word_similarity)
                    list_of_words.append(row_of_similarity)
                    target_words_with_min_max.append(output_line)

                Tools.print_training_time(total_training)

# Pairs

In [ ]:
import os
import numpy as np
from contextlib import redirect_stdout
from tqdm import tqdm
from time import time
from collections import defaultdict
from tmu.models.autoencoder.autoencoder import TMAutoEncoder
from sklearn.metrics.pairwise import cosine_similarity
from evaluation import Evaluation
from tools import Tools
from directories import Dicrectories
import cProfile

clause_weight_threshold = 10
clause_drop_p = 0.0
clauses = 80
factor = 20
T = factor*40
s = 5.0
epochs = 100
number_of_examples = 48
accumulation = 10
sub_accumulation = 10
top_max_clauses1 = 0
top_max_clauses2 = 0
with_clause_update = True

eval = Evaluation()
def preprocess_text(text):
    return text
vectorizer_X = Tools.read_pickle_data("vectorizer_X.pickle")
feature_names = vectorizer_X.get_feature_names_out()
number_of_features = vectorizer_X.get_feature_names_out().shape[0]

for dataset_name in os.listdir(Dicrectories.datasets):
    if dataset_name == 'rg-65':
        current_folder_path = os.path.join(Dicrectories.datasets, dataset_name)
        if os.path.isdir(current_folder_path):
            files_start_name = os.path.join(current_folder_path, dataset_name)
            pair_list = Tools.get_dataset_pairs(files_start_name + '.csv')
            output_active, target_words = Tools.get_targets(files_start_name)
            available_pair_list = []
            pairs_output_active = []
            for pair, score in pair_list:
                word1, word2 = pair[0], pair[1]
                if all(word in target_words for word in [word1, word2]):
                    available_pair_list.append([pair,score])
                    pairs_output_active.append([vectorizer_X.vocabulary_[word1], vectorizer_X.vocabulary_[word2]])
            
            result_filepath = Dicrectories.test(dataset_name,"pair_phase2")
            with open(result_filepath, 'w') as file, redirect_stdout(file):
                total_training = 0
                print("Epochs: %d" % epochs)
                print("Target words: %d" % len(target_words))
                print("No of features: %d" % number_of_features)
                print("Clauses: %d" % clauses)
                print("with_clause_update: %s" % with_clause_update)
                print("Examples: %d" % number_of_examples)
                print("Accumulation: %d" % accumulation)
                print("Sub Accumulation: %d" % sub_accumulation)
                print("top_max_clauses1: %d" % top_max_clauses1)
                print("top_max_clauses2: %d\n" % top_max_clauses2)
                
                epochs_progress_bar = tqdm(total=epochs, desc="Running Epochs")
                for e in range(epochs):
                    print("\nEpoch #: %d" % e)
                    epoch_time = 0
                    target_similarity=defaultdict(list)
                    for pair_index, pair in enumerate(pairs_output_active):
                        pair_output_active = np.empty(2, dtype=np.uint32)
                        pair_output_active[0] = pair[0]
                        pair_output_active[1] = pair[1]
                        start_training = time()
                        tm = TMAutoEncoder(clauses, T, s, pair_output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=0.5)

                        # profile = cProfile.Profile() 
                        # profile.enable()
                        tm.knowledge_fit(
                            number_of_examples = number_of_examples,
                            number_of_features = number_of_features,
                            sub_accumulation = sub_accumulation,
                            top_max_clauses1 = top_max_clauses1,
                            top_max_clauses2 = top_max_clauses2,
                            with_clause_update = with_clause_update,
                            print_c = False
                            )
                        # profile.disable()
                        # profile.print_stats(sort='time')

                        stop_training = time()
                        pair_time = stop_training - start_training
                        epoch_time = epoch_time + pair_time

                        profile = np.empty((2, clauses))
                        weights = tm.get_weights(0)
                        profile[0,:] = np.where(weights >= clause_weight_threshold, weights, 0)
                        weights = tm.get_weights(1)
                        profile[1,:] = np.where(weights >= clause_weight_threshold, weights, 0)
                        similarity = cosine_similarity(profile)

                        sorted_index = np.argsort(-1*similarity[0,:])
                        target_similarity[available_pair_list[pair_index][0]]  = similarity[0,sorted_index[1]]

                    Tools.print_training_time(epoch_time)
                    total_training = total_training + epoch_time
                    eval.calculate(target_similarity,available_pair_list)
                    epochs_progress_bar.update(1)
                epochs_progress_bar.close()
                Tools.print_training_time(total_training)

# Epochs then pairs

In [ ]:
import os
import numpy as np
from contextlib import redirect_stdout
from tqdm import tqdm
from time import time
from collections import defaultdict
from tmu.models.autoencoder.autoencoder import TMAutoEncoder
from sklearn.metrics.pairwise import cosine_similarity
from evaluation import Evaluation
from tools import Tools
from directories import Dicrectories
import cProfile

clause_weight_threshold = 0
number_of_examples = 1000
clause_drop_p = 0.0
factor = 20
clauses = 80
T = factor*40
s = 5.0
accumulation = 10
epochs = 100

eval = Evaluation()
def preprocess_text(text):
    return text
vectorizer_X = Tools.read_pickle_data("vectorizer_X.pickle")
feature_names = vectorizer_X.get_feature_names_out()
number_of_features = vectorizer_X.get_feature_names_out().shape[0]

for dataset_name in os.listdir(Dicrectories.datasets):
    if dataset_name == 'rg-65':
        current_folder_path = os.path.join(Dicrectories.datasets, dataset_name)
        if os.path.isdir(current_folder_path):
            files_start_name = os.path.join(current_folder_path, dataset_name)
            pair_list = Tools.get_dataset_pairs(files_start_name + '.csv')
            output_active, target_words = Tools.get_targets(files_start_name)
            available_pair_list = []
            pairs_output_active = []
            for pair, score in pair_list:
                word1, word2 = pair[0], pair[1]
                if all(word in target_words for word in [word1, word2]):
                    available_pair_list.append([pair,score])
                    pairs_output_active.append([vectorizer_X.vocabulary_[word1], vectorizer_X.vocabulary_[word2]])
            
            result_filepath = Dicrectories.test(dataset_name,"pair_cross_phase2")
            with open(result_filepath, 'w') as file, redirect_stdout(file):
                total_training = 0
                print("Epochs: %d" % epochs)
                print("Target words: %d" % len(target_words))
                print("Accumulation: %d" % accumulation)
                print("Examples: %d" % number_of_examples)
                print("No of features: %d" % number_of_features)
                print("Clauses: %d" % clauses)
                
                target_similarity=defaultdict(list)
                pairs_progress_bar = tqdm(total=len(pairs_output_active), desc="Running Pairs")
                for pair_index, pair in enumerate(pairs_output_active):
                    print("\nPair #: %d" % (pair_index+1))
                    pair_output_active = np.empty(2, dtype=np.uint32)
                    pair_output_active[0] = pair[0]
                    pair_output_active[1] = pair[1]
                    tm = TMAutoEncoder(clauses, T, s, pair_output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=0.5)

                    pair_time = 0
                    for e in range(epochs):
                        start_training = time()
                        # print("Epoch #: %d" % (e + 1))
                        # profile = cProfile.Profile() 
                        # profile.enable()
                        tm.knowledge_fit(
                            number_of_examples = number_of_examples,
                            number_of_features = number_of_features,
                            top_max_clauses1 = 0,
                            top_max_clauses2 = 2,
                            with_clause_update = False,
                            print_c = False
                            )
                        # profile.disable()
                        # profile.print_stats(sort='time')

                        stop_training = time()
                        epoch_time = stop_training - start_training
                        pair_time = epoch_time + pair_time
                
                    profile = np.empty((2, clauses))
                    weights = tm.get_weights(0)
                    profile[0,:] = np.where(weights >= clause_weight_threshold, weights, 0)
                    weights = tm.get_weights(1)
                    profile[1,:] = np.where(weights >= clause_weight_threshold, weights, 0)
                    similarity = cosine_similarity(profile)
                    sorted_index = np.argsort(-1*similarity[0,:])
                    target_similarity[available_pair_list[pair_index][0]]  = similarity[0,sorted_index[1]]
                    
                    Tools.print_training_time(pair_time)
                    total_training = total_training + pair_time
                    pairs_progress_bar.update(1)

                eval.calculate(target_similarity,available_pair_list)
                pairs_progress_bar.close()
                Tools.print_training_time(total_training)

# All with skip

In [ ]:
import os
import numpy as np
from contextlib import redirect_stdout
from tqdm import tqdm
from time import time
from collections import defaultdict
from tmu.models.autoencoder.autoencoder import TMAutoEncoder
from sklearn.metrics.pairwise import cosine_similarity
from evaluation import Evaluation
from tools import Tools
from directories import Dicrectories

target_similarity=defaultdict(list)
clause_weight_threshold = 0
number_of_examples = 1000
clause_drop_p = 0.0
factor = 20
clauses = 80
T = factor*40
s = 5.0
accumulation = 10
epochs = 1000
max_spearman = 0.9
selection_size = 100
top_max_clauses1 = 5
top_max_clauses2 = 5
with_clause_update = False

eval = Evaluation()
def preprocess_text(text):
    return text
vectorizer_X = Tools.read_pickle_data("vectorizer_X.pickle")
feature_names = vectorizer_X.get_feature_names_out()
number_of_features = vectorizer_X.get_feature_names_out().shape[0]

for dataset_name in os.listdir(Dicrectories.datasets):
    if dataset_name == 'mturk-771':
        current_folder_path = os.path.join(Dicrectories.datasets, dataset_name)
        if os.path.isdir(current_folder_path):
            files_start_name = os.path.join(current_folder_path, dataset_name)

            result_filepath = Dicrectories.test(dataset_name,"skip_all_phase2")
            with open(result_filepath, 'w') as file, redirect_stdout(file):
                all_pair_list = Tools.get_dataset_pairs(files_start_name + '.csv')
                for part in range(0, len(all_pair_list), selection_size):
                    pair_list = all_pair_list[part:part+selection_size]
                    
                    word1_list=[]
                    word2_list=[]    
                    for (x,y) in pair_list:
                        (word1, word2) = x
                        word1_list.append(word1)
                        word2_list.append(word2)
                
                    word_total= list(set(word1_list + word2_list))
                    target_words=[]
                    for i in word_total:
                        if i in vectorizer_X.vocabulary_:
                            target_words.append(i)
                            
                    output_active = np.empty(len(target_words), dtype=np.uint32)
                    for i in range(len(target_words)):
                        target_word = target_words[i]
                        target_id = vectorizer_X.vocabulary_[target_word]
                        output_active[i] = target_id

                    tm = TMAutoEncoder(clauses, T, s, output_active, max_included_literals=3, accumulation=accumulation, feature_negation=False, platform='CPU', output_balancing=0.5)
                    total_training = 0
                    print("Epochs: %d" % epochs)
                    print("Target words: %d" % len(target_words))
                    print("Accumulation: %d" % accumulation)
                    print("No of features: %d" % number_of_features)
                    print("Clauses: %d\n" % clauses)
                    print("with_clause_update: %s\n" % with_clause_update)
                    print("Examples: %d" % number_of_examples)
                    print("top_max_clauses1: %d\n" % top_max_clauses1)
                    print("top_max_clauses2: %d\n" % top_max_clauses2)
                    
                    epochs_progress_bar = tqdm(total=epochs, desc="Running Epochs")
                    for e in range(epochs):
                        print("\nEpoch #: %d" % e)
                        start_training = time()
                        tm.knowledge_fit(
                            number_of_examples = number_of_examples,
                            number_of_features = number_of_features,
                            top_max_clauses1 = top_max_clauses1,
                            top_max_clauses2 = top_max_clauses2,
                            with_clause_update = with_clause_update,
                            print_c = False
                            )
                        stop_training = time()
                        epoch_time = stop_training - start_training
                        Tools.print_training_time(epoch_time)
                        total_training = total_training + epoch_time
    
                        profile = np.empty((len(target_words), clauses))
                        for i in range(len(target_words)):
                            weights = tm.get_weights(i)
                            profile[i,:] = np.where(weights >= clause_weight_threshold, weights, 0)
                        similarity = cosine_similarity(profile)
                        for i in range(len(target_words)):
                            sorted_index = np.argsort(-1*similarity[i,:])
                            for j in range(1, len(target_words)):
                                target_similarity[(target_words[i], target_words[sorted_index[j]])]  = similarity[i,sorted_index[j]]
                       
                        spearman = eval.calculate(target_similarity,pair_list)
                        if spearman > max_spearman:
                            break
                        epochs_progress_bar.update(1)
                    epochs_progress_bar.close()
    
                    print("\n=====================================\nClauses\n=====================================")
                    for j in range(clauses):
                        print("Clause #%-2d " % (j), end=' ')
                        for tw in range(len(target_words)):
                            print("%s:W%-5d " % (target_words[tw], tm.get_weight(tw, j)), end='| ')
                        l = [] 
                        number_of_literals = 0 
                        for k in range(tm.clause_bank.number_of_literals):
                            if tm.get_ta_action(j, k) == 1:
                                number_of_literals = number_of_literals + 1
                                if k < tm.clause_bank.number_of_features:
                                    l.append("%s(%d)" % (feature_names[k], tm.clause_bank.get_ta_state(j, k)))
                                else:
                                    l.append("¬%s(%d)" % (feature_names[k-tm.clause_bank.number_of_features], tm.clause_bank.get_ta_state(j, k)))
                        print(": No of features:%-6d" % (number_of_literals), end=" ==> ")
                        try:
                            print(" - ".join(l))
                        except UnicodeEncodeError:
                            print(" exception ")
                    
                    print("\n=====================================\nWord Similarity\n=====================================")
                    max_word_length = len(max(target_words, key=len))
                    list_of_words = []
                    target_words_with_min_max = []
                    for i in range(len(target_words)):
                        row_of_similarity = []
                        sorted_index = np.argsort(-1*similarity[i,:])
                        min_similarity = 1.0
                        max_similarity = 0.0
                        word_similarity = []
                        for j in range(1, len(target_words)):
                            target_similarity[(target_words[i], target_words[sorted_index[j]])]  = similarity[i,sorted_index[j]]
                            row_of_similarity.append(target_words[sorted_index[j]])
                            word_similarity.append("{:<{}}({:.2f})  ".format(target_words[sorted_index[j]], max_word_length, similarity[i, sorted_index[j]]))
                            if(min_similarity > similarity[i,sorted_index[j]]):
                                min_similarity = similarity[i,sorted_index[j]]
                            if(max_similarity < similarity[i,sorted_index[j]]):
                                max_similarity = similarity[i,sorted_index[j]]
                    
                        output_line = f"{target_words[i]:<{max_word_length}}: Min:{min_similarity:.2f}, Max:{max_similarity:.2f}"
                        print(output_line, end='     ==> ')
                        print(word_similarity)
                        list_of_words.append(row_of_similarity)
                        target_words_with_min_max.append(output_line)
    
                    Tools.print_training_time(total_training)

                spearman = eval.calculate(target_similarity,all_pair_list)
                dir_name, old_file_name = os.path.split(result_filepath)
                new_file_path = os.path.join(dir_name, "{:.2f}".format(spearman) + "_"  + old_file_name)
                os.rename(result_filepath, new_file_path)

# All with Groupping

In [ ]:
grouped_pairs = {}
for pair in pair_list:
    first_word = pair[0][0]
    if first_word in grouped_pairs:
        grouped_pairs[first_word].append(pair)
    else:
        grouped_pairs[first_word] = [pair]

for key, value in grouped_pairs.items():
    print(key + ':')
    for pair in value:
        print(pair)
    print()

In [ ]:
from tools import Tools
num_cached = Tools.read_pickle_data.cache_info().currsize
print("Number of cached results:", num_cached)

In [ ]:
import random
for dataset_name in os.listdir(Dicrectories.datasets):
    if dataset_name == 'rg-65':
        current_folder_path = os.path.join(Dicrectories.datasets, dataset_name)
        if os.path.isdir(current_folder_path):
            files_start_name = os.path.join(current_folder_path, dataset_name)

            pair_list = Tools.get_dataset_pairs(files_start_name + '.csv')
            output_active, target_words = Tools.get_targets(files_start_name)

            number_of_clauses = len(output_active)
            class_index = np.arange(number_of_clauses, dtype=np.uint32)
            knowledge_directory = Dicrectories.knowledge
            rng = np.random.RandomState(None)
            rng.shuffle(class_index)

            for index in class_index:
                tw = output_active[index]
                target_value = random.randint(0, 1)

                tw_knowledge_path = Dicrectories.pickle_by_id(knowledge_directory , tw)
                tw_all_clauses = Tools.read_pickle_data(tw_knowledge_path)
                if target_value == 1:
                    tw_filtered_clauses = [clause for clause in tw_all_clauses if clause[0] > 0]
                else:
                    tw_filtered_clauses = [clause for clause in tw_all_clauses if clause[0] < 0]

                tw_clauses_subset = random.sample(tw_filtered_clauses, accumulation)
                if(top_max_clauses1 > 0):
                    tw_clauses_subset = sorted(tw_clauses_subset, key=lambda x: x[0], reverse=True)[:top_max_clauses1]

                documents_of_features = []
                for tw_clause in tw_clauses_subset:
                    related_literals = tw_clause[1]
                    for literal in related_literals:
                        documents_of_features.append(literal)
                        literal_knowledge_path = Dicrectories.pickle_by_id(knowledge_directory , literal)
                        literal_all_clauses = Tools.read_pickle_data(literal_knowledge_path)
                        if target_value == 1:
                            literal_filtered_clauses = [clause for clause in literal_all_clauses if clause[0] > 0]
                        else:
                            literal_filtered_clauses = [clause for clause in literal_all_clauses if clause[0] < 0]
                        
                        literal_clauses_subset = random.sample(literal_filtered_clauses, sub_accumulation)
                        if(top_max_clauses2 > 0):
                            literal_clauses_subset = sorted(literal_clauses_subset, key=lambda x: x[0], reverse=True)[:top_max_clauses2]

                        for literal_clause in literal_clauses_subset:
                            literals = literal_clause[1]
                            for sub_literal in literals:
                                documents_of_features.append(sub_literal)
            print(documents_of_features[0])

            